# 01 · Mathematical foundations

Gaussian integration by parts turns a derivative of `E[φ(S_T)]` into `E[φ(S_T)·π]`. Here we check the weights *without simulation*, by quadrature, and look at why they work for an indicator payoff. Derivations: `docs/mathematical_derivations.md`.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from mc_greeks import *
from mc_greeks.estimators import exact_estimator_moments, per_path_function
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLES = ROOT / "results" / "tables"
pd.set_option("display.float_format", lambda v: f"{v:.6g}")

## Exact check: E[D φ(S_T) π] equals the closed-form Greek
`exact_estimator_moments` integrates the per-path term against the normal density, splitting at the payoff's jumps.

In [2]:
p, f = PAPER_PARAMS, PAPER_PAYOFF
rows = []
for g in ['delta', 'gamma', 'vega']:
    m = exact_estimator_moments('Malliavin', g, f, p)
    rows.append({'greek': g, 'E[D phi pi] (quadrature)': m['mean'], 'analytical': m['truth'],
                 'per-path variance': m['var1']})
pd.DataFrame(rows)

,greek,E[D phi pi] (quadrature),analytical,per-path variance
0,delta,-0.00133486,-0.00133486,1.49473e-05
1,gamma,-0.000388757,-0.000388757,6.66825e-07
2,vega,-0.777514,-0.777514,2.6673


## The weight does not depend on the payoff
The same `π_Δ = W_T/(xσT)` gives the Delta of a call, a digital and the interval digital:

In [3]:
for pay in [EuropeanCall(100), CashDigitalCall(100), PAPER_PAYOFF]:
    m = exact_estimator_moments('Malliavin', 'delta', pay, p)
    print(f"{pay.name:9s} quadrature {m['mean']:+.8f}   analytical {m['truth']:+.8f}")

call      quadrature +0.72574688   analytical +0.72574688
digital   quadrature +0.01666123   analytical +0.01666123
interval  quadrature -0.00133486   analytical -0.00133486


## Why finite differences struggle with a jump
Per-path CRN central difference for the digital: it is 0 except on an O(h) event where it equals 1/(2h). N·Var therefore grows like 1/h.

In [4]:
d = CashDigitalCall(100)
for h in [2.0, 0.5, 0.1, 0.02]:
    m = exact_estimator_moments('FD-CRN-ctr', 'delta', d, p, h=h)
    print(f'h={h:5}:  bias={m["bias"]:+.2e}   N*Var={m["var1"]:.3e}   h*N*Var={h*m["var1"]:.3e}')

h=  2.0:  bias=-1.45e-05   N*Var=3.489e-03   h*N*Var=6.977e-03
h=  0.5:  bias=-9.03e-07   N*Var=1.480e-02   h*N*Var=7.399e-03
h=  0.1:  bias=-3.61e-08   N*Var=7.510e-02   h*N*Var=7.510e-03
h= 0.02:  bias=-1.44e-09   N*Var=3.766e-01   h*N*Var=7.532e-03


The last column is roughly constant, confirming the O(1/h) variance law.